In [ ]:
%pip install scipy
# %pip install dataclasses
%pip install pandas
%pip install numpy
%pip install torch
%pip install wandb
%pip install datasets
%pip install transformers
%pip install huggingface_hub
%pip install evaluate
%pip install torchaudio
# %pip install librosa

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s eta 0:00:00


In [ ]:
# %%
import os
from datetime import datetime
from typing import Any, Dict, List, Optional

import numpy as np
import torch
import torch.nn.functional as F
import torchaudio.transforms as T
import wandb
import evaluate

from datasets import load_dataset, Audio, load_from_disk, concatenate_datasets
from transformers import (
    AutoModelForAudioClassification,
    AutoFeatureExtractor,
    AutoConfig,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    EarlyStoppingCallback,
    get_cosine_schedule_with_warmup,
)
from torch.optim import AdamW
from huggingface_hub import login
from google.colab import userdata
import functools, random

current_time_str = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"Current time: {current_time_str}")

# ──────────────────────────────────────────────
# GPU CHECK
# ──────────────────────────────────────────────
print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")
print(f"torch.cuda.get_device_name(): {torch.cuda.get_device_name()}")

# ──────────────────────────────────────────────
# LOGIN
# ──────────────────────────────────────────────
HF_TOKEN = userdata.get('HF_TOKEN')
if not HF_TOKEN:
    raise ValueError("HF_TOKEN not set")
login(token=HF_TOKEN)
print("Logged in to Hugging Face!")

WANDB_TOKEN = userdata.get('WANDB_TOKEN')
if not WANDB_TOKEN:
    raise ValueError("WANDB_TOKEN not set")
wandb.login(key=WANDB_TOKEN)

# ──────────────────────────────────────────────
# HYPERPARAMETERS
# ──────────────────────────────────────────────

model_id                    = "utter-project/mHuBERT-147"
batch_size                  = 64
gradient_accumulation_steps = 2
num_train_epochs            = 20
unfreeze_start_step         = 800
unfreeze_every_steps        = 70
max_duration                = 5        # seconds
label_smoothing             = 0.1 #0.2
weight_decay                = 0.05
sampling_rate               = 16000
warmup_ratio                = 0.2
backbone_lr                 = 1e-6 #3e-6
head_lr                     = 0.001
early_stopping_patience     = 4
max_grad_norm               = 0.5

do_apply_dropout     = True
hidden_dropout       = 0.1
attention_dropout    = 0.01
activation_dropout   = 0.01
feat_proj_dropout    = 0.1

# Augmentation settings
AUG_PROB     = 0.6    # per-sample probability of applying augmentation
# pitch is pre-baked (slow GPU op, done once); speed+noise applied live in collator
PITCH_SEMITONES_RANGE = (-3.0, 3.0)   # semitones up/down
SPEED_RATE_RANGE      = (0.85, 1.15)  # speed factor range
NOISE_SNR_RANGE       = (15.0, 40.0)  # SNR dB range

input_features_key = "input_values"   # mHuBERT uses input_values

AVAILABLE_CPU = userdata.get("REQUEST_CPU")
if not AVAILABLE_CPU:
    import multiprocessing
    AVAILABLE_CPU = multiprocessing.cpu_count()
else:
    AVAILABLE_CPU = int(AVAILABLE_CPU)

# GPU for pitch preprocessing (20-50x faster than CPU)
_AUG_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Augmentation device: {_AUG_DEVICE}")

WANDB_TEAM    = userdata.get("WANDB_TEAM")
WANDB_PROJECT = userdata.get("WANDB_PROJECT")

# ──────────────────────────────────────────────
# WANDB INIT
# ──────────────────────────────────────────────
wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_TEAM,
    name=f"SLID_aug_{model_id.split('/')[-1]}_{current_time_str}",
    config={
        "model_id":                    model_id,
        "batch_size":                  batch_size,
        "gradient_accumulation_steps": gradient_accumulation_steps,
        "effective_batch_size":        batch_size * gradient_accumulation_steps, # Полезно для сравнения ранов
        "num_train_epochs":            num_train_epochs,
        "backbone_lr":                 backbone_lr,
        "head_lr":                     head_lr,
        "max_duration":                max_duration,
        "unfreeze_start_step":         unfreeze_start_step,
        "unfreeze_every_steps":        unfreeze_every_steps,
        "label_smoothing":             label_smoothing,
        "weight_decay":                weight_decay,
        "warmup_ratio":                warmup_ratio,
        "max_grad_norm":               max_grad_norm,         # Добавили для контроля стабильности
        "hidden_dropout":              hidden_dropout,        # Важно видеть 0.1 в логах
        "aug_strategy":                "pitch+speed+noise",
        "aug_prob":                    AUG_PROB,
    }
)

# ──────────────────────────────────────────────
# FEATURE EXTRACTOR & DATA
# ──────────────────────────────────────────────
feature_extractor = AutoFeatureExtractor.from_pretrained(
    model_id, do_normalize=True, return_attention_mask=True
)

dataset  = load_dataset("badrex/nnti-dataset-full")
print(f"dataset['train']: {dataset['train']}")
print(f"dataset['train'][0]: {dataset['train'][0]}")

train_ds = dataset['train'].shuffle(seed=42).cast_column(
    "audio_filepath", Audio(sampling_rate=sampling_rate))
valid_ds = dataset['validation'].shuffle(seed=42).cast_column(
    "audio_filepath", Audio(sampling_rate=sampling_rate))

LABELS        = train_ds.unique('language')
sorted_labels = sorted(l.upper() for l in LABELS)
print(f"Languages: {sorted_labels}")
str_to_int    = {s: i for i, s in enumerate(LABELS)}
int_to_str    = {i: s for s, i in str_to_int.items()}
num_labels    = len(LABELS)

# ──────────────────────────────────────────────
# AUGMENTATION FUNCTIONS
#
# SLOW (pitch)  → pre-baked into dataset cache on GPU, done once
# FAST (speed, noise) → applied live in collator, no GPU stall during training
# ──────────────────────────────────────────────

def _to_tensor(arr: np.ndarray) -> torch.Tensor:
    return torch.from_numpy(arr.astype(np.float32)).unsqueeze(0)

def _to_numpy(t: torch.Tensor) -> np.ndarray:
    return t.squeeze(0).numpy()


def augment_pitch(arr: np.ndarray) -> np.ndarray:
    """GPU pitch shift — runs during preprocessing, cached to disk."""
    n_steps = random.uniform(*PITCH_SEMITONES_RANGE)
    with torch.no_grad():
        x       = _to_tensor(arr).to(_AUG_DEVICE)
        shifted = T.PitchShift(sample_rate=sampling_rate, n_steps=n_steps).to(_AUG_DEVICE)(x)
    return shifted.cpu().squeeze(0).numpy()


# Создаем кэш ресэмплеров ОДИН РАЗ в глобальной области видимости
SPEED_RATES = [0.85, 0.90, 0.95, 1.05, 1.10, 1.15]
RESAMPLERS = {}
for rate in SPEED_RATES:
    new_sr = int(sampling_rate * rate)
    # T.Resample создается только один раз для каждой скорости
    RESAMPLERS[rate] = T.Resample(orig_freq=new_sr, new_freq=sampling_rate)

def augment_speed(arr: np.ndarray) -> np.ndarray:
    """Оптимизированный speed perturbation с использованием кэша."""
    rate = random.choice(SPEED_RATES) # Выбираем из готовых
    resampler = RESAMPLERS[rate]      # Берем готовый фильтр

    resampled = resampler(_to_tensor(arr))
    orig_len  = arr.shape[0]

    if resampled.shape[1] < orig_len:
        resampled = F.pad(resampled, (0, orig_len - resampled.shape[1]))
    else:
        resampled = resampled[:, :orig_len]
    return _to_numpy(resampled)


def augment_noise(arr: np.ndarray) -> np.ndarray:
    """Additive Gaussian noise — near-zero cost, runs live in collator."""
    snr_db    = random.uniform(*NOISE_SNR_RANGE)
    x         = torch.from_numpy(arr.astype(np.float32))
    sig_pow   = x.pow(2).mean().clamp(min=1e-9)
    noise_pow = sig_pow / (10 ** (snr_db / 10))
    return (x + torch.randn_like(x) * noise_pow.sqrt()).numpy()


# Fast augmentations applied live in the collator
LIVE_AUG_FUNCTIONS = [augment_speed, augment_noise]

# ──────────────────────────────────────────────
# PREPROCESSING  (with pitch pre-baking)
#
# Cache layout:
#   ./processed_data/mHuBERT-147_5sec/original/  ← clean audio
#   ./processed_data/mHuBERT-147_5sec/pitch/     ← pitch-shifted copy
#   valid same structure (original only)
#
# Training set = original + pitch copy (2× data)
# ──────────────────────────────────────────────
keep_cols = ['speaker_id', 'language']

def preprocess_function(examples, aug_fn=None):
    arrays = [x["array"] for x in examples["audio_filepath"]]
    if aug_fn is not None:
        augmented = []
        for a in arrays:
            try:
                augmented.append(aug_fn(a))
            except Exception:
                print('Got exception in preprocess_function')
                augmented.append(a)   # fallback to original on error
        arrays = augmented

    inputs = feature_extractor(
        arrays,
        sampling_rate=feature_extractor.sampling_rate,
        truncation=True,
        max_length=int(feature_extractor.sampling_rate * max_duration),
        return_attention_mask=True,
    )
    inputs["label"]  = [str_to_int[x] for x in examples["language"]]
    inputs[input_features_key] = [np.array(x) for x in inputs[input_features_key]]
    inputs["length"] = [len(f) for f in inputs[input_features_key]]
    return inputs


cache_base = f"./processed_data/{model_id.split('/')[-1]}_{max_duration}sec"

def load_or_create(ds, split_name: str, aug_name: str, aug_fn=None):
    """Load from disk if cached, otherwise map and save."""
    path     = f"{cache_base}/{split_name}/{aug_name}"
    # GPU pitch aug must use num_proc=1 to avoid CUDA multi-process conflicts
    num_proc = 1 if (aug_fn == augment_pitch) else AVAILABLE_CPU
    bs       = 64 if (aug_fn == augment_pitch) else 32   # larger batch amortises GPU launch

    if os.path.exists(path):
        print(f"  Loading cached {split_name}/{aug_name} from {path}")
        return load_from_disk(path)

    print(f"  Preprocessing {split_name}/{aug_name} "
          f"({'GPU' if aug_fn == augment_pitch else 'CPU'})…")
    result = ds.map(
        functools.partial(preprocess_function, aug_fn=aug_fn),
        batched=True,
        batch_size=bs,
        num_proc=num_proc,
        remove_columns=[c for c in ds.column_names if c not in keep_cols],
    )
    result.save_to_disk(path)
    return result


# Build train: original + pitch-shifted copy
orig_train  = load_or_create(train_ds, 'train', 'original', aug_fn=None)
pitch_train = load_or_create(train_ds, 'train', 'pitch',    aug_fn=augment_pitch)
train_ds_encoded = concatenate_datasets([orig_train, pitch_train]).shuffle(seed=42)
print(f"Train size after pitch augmentation: {len(train_ds_encoded)} "
      f"(was {len(orig_train)})")

# Validation: original only — never augmented for fair evaluation
valid_ds_encoded = load_or_create(valid_ds, 'valid', 'original', aug_fn=None)

# ──────────────────────────────────────────────
# COLLATOR  (speed + noise applied live)
# ──────────────────────────────────────────────

class AugmentingCollator:
    """
    Applies speed perturbation and/or noise on-the-fly during training.
    Eval collator (training=False) just pads with no augmentation.
    """
    def __init__(self, aug_prob: float = 0.5, training: bool = True):
        self.aug_prob = aug_prob
        self.training = training

    def _maybe_augment(self, arr: np.ndarray) -> np.ndarray:
        if not self.training or random.random() > self.aug_prob:
            return arr
        try:
            return random.choice(LIVE_AUG_FUNCTIONS)(arr)
        except Exception:
            print('Exception in AugmentingCollator._maybe_augment')
            return arr

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        augmented = [self._maybe_augment(np.array(f[input_features_key]))
                     for f in features]

        # Re-normalise after augmentation
        normed = []
        for a in augmented:
            m, s = a.mean(), a.std()
            normed.append(((a - m) / (s + 1e-7)).astype(np.float32))

        batch = feature_extractor.pad(
            {input_features_key: normed,
             "attention_mask": [f["attention_mask"] for f in features]},
            padding=True, return_tensors="pt",
        )
        batch["labels"] = torch.tensor(
            [f["label"] for f in features], dtype=torch.long)
        return batch


train_collator = AugmentingCollator(aug_prob=AUG_PROB, training=True)
eval_collator  = AugmentingCollator(aug_prob=0.0,      training=False)

# ──────────────────────────────────────────────
# MODEL
# ──────────────────────────────────────────────
config = AutoConfig.from_pretrained(model_id)
config.num_labels = num_labels
config.label2id   = str_to_int
config.id2label   = int_to_str
if do_apply_dropout:
    config.hidden_dropout     = hidden_dropout
    config.attention_dropout  = attention_dropout
    config.activation_dropout = activation_dropout
    config.feat_proj_dropout  = feat_proj_dropout

slid_model = AutoModelForAudioClassification.from_pretrained(model_id, config=config)

# Gradient checkpointing saves ~40% VRAM on T4/A100 (costs ~30% speed)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
if vram_gb < 20:
    print(f"⚠️ Low VRAM ({vram_gb:.1f}GB). Enabling Gradient Checkpointing...")
    slid_model.gradient_checkpointing_enable()
else:
    print(f"🚀 High VRAM ({vram_gb:.1f}GB). Disabling Checkpointing for speed boost!")

# Freeze backbone initially — only head trains until unfreeze_at_step
for param in slid_model.hubert.parameters():
    param.requires_grad = False

for param in slid_model.hubert.feature_extractor.parameters():
    param.requires_grad = False

# ──────────────────────────────────────────────
# OPTIMIZER & SCHEDULER
# ──────────────────────────────────────────────
def get_optimizer_and_scheduler(model, num_training_steps):
    backbone_params = [p for n, p in model.named_parameters()
                       if "classifier" not in n and "projector" not in n]
    head_params     = [p for n, p in model.named_parameters()
                       if "classifier" in n or "projector" in n]

    optimizer = AdamW([
        {"params": backbone_params, "lr": backbone_lr},
        {"params": head_params,     "lr": head_lr},
    ], weight_decay=weight_decay)

    num_warmup_steps = int(warmup_ratio * num_training_steps)
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps,
    )
    return optimizer, scheduler

# ──────────────────────────────────────────────
# TRAINING ARGS
# ──────────────────────────────────────────────
steps_per_epoch   = len(train_ds_encoded) // (batch_size * gradient_accumulation_steps)
num_training_steps = steps_per_epoch * num_train_epochs
print(f"Steps per epoch: {steps_per_epoch} | Total steps: {num_training_steps}")

optimizer, scheduler = get_optimizer_and_scheduler(slid_model, num_training_steps)

training_args = TrainingArguments(
    output_dir="./indic-SLID/checkpoints",
    group_by_length=True,
    dataloader_num_workers=min(4, AVAILABLE_CPU),  # capped: more workers → diminishing returns
    dataloader_pin_memory=True,
    report_to="wandb",
    logging_steps=10,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    gradient_accumulation_steps=gradient_accumulation_steps,
    num_train_epochs=num_train_epochs,
    weight_decay=weight_decay,
    warmup_steps=int(warmup_ratio * num_training_steps),
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    save_total_limit=2,
    fp16=True,
    max_grad_norm=max_grad_norm,
    push_to_hub=False,
    label_smoothing_factor=label_smoothing,
    remove_unused_columns=False,   # keeps all columns (collator handles selection)
)

# ──────────────────────────────────────────────
# METRICS & CALLBACKS
# ──────────────────────────────────────────────
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return accuracy_metric.compute(
        predictions=predictions, references=eval_pred.label_ids)


class GradualUnfreezeCallback(TrainerCallback):
    """
    Размораживает backbone послойно начиная с unfreeze_start_step,
    добавляя по одному transformer layer каждые unfreeze_every_steps шагов.
    Это плавнее чем разморозить всё сразу.
    """
    def __init__(self, unfreeze_start_step: int, unfreeze_every_steps: int = 100):
        self.unfreeze_start_step  = unfreeze_start_step
        self.unfreeze_every_steps = unfreeze_every_steps
        self.layers_unfrozen      = 0

    def on_step_begin(self, args, state, control, model=None, **kwargs):
        step = state.global_step
        if step < self.unfreeze_start_step:
            return

        steps_since_start = step - self.unfreeze_start_step
        target_unfrozen   = steps_since_start // self.unfreeze_every_steps + 1

        layers = model.hubert.encoder.layers   # list of transformer layers
        if self.layers_unfrozen < target_unfrozen <= len(layers):
            # Unfreeze from top (closest to head) downward
            layer_idx = len(layers) - target_unfrozen
            for param in layers[layer_idx].parameters():
                param.requires_grad = True
            self.layers_unfrozen = target_unfrozen
            print(f"✓ Unfrozen layer {layer_idx} at step {step} "
                  f"({self.layers_unfrozen}/{len(layers)} layers active)")

        # Finally unfreeze feature extractor and positional embeddings
        if self.layers_unfrozen >= len(layers):
            for param in model.hubert.parameters():
                param.requires_grad = True

# ──────────────────────────────────────────────
# TRAINER
# ──────────────────────────────────────────────
trainer = Trainer(
    slid_model,
    training_args,
    train_dataset=train_ds_encoded,
    eval_dataset=valid_ds_encoded,
    processing_class=feature_extractor,
    data_collator=train_collator,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, scheduler),
    callbacks=[
        GradualUnfreezeCallback(
            unfreeze_start_step=unfreeze_start_step,
            unfreeze_every_steps=unfreeze_every_steps,
        ),
        EarlyStoppingCallback(early_stopping_patience=early_stopping_patience),
    ],
)

# Use no-augmentation collator during eval
original_get_eval_dataloader = trainer.get_eval_dataloader
def get_eval_dataloader_no_aug(eval_dataset=None):
    trainer.data_collator = eval_collator
    dl = original_get_eval_dataloader(eval_dataset)
    trainer.data_collator = train_collator
    return dl
trainer.get_eval_dataloader = get_eval_dataloader_no_aug

# ──────────────────────────────────────────────
# TRAIN
# ──────────────────────────────────────────────
print(f"Model is on device: {slid_model.device}")
wandb.watch(slid_model, log="gradients", log_freq=100)

print("Train loop starting...")
#trainer.train()

# ──────────────────────────────────────────────
# EVALUATE & LOG
# ──────────────────────────────────────────────
print("Final evaluation starting...")
trainer.evaluate()

predictions = trainer.predict(valid_ds_encoded)
preds  = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

wandb.log({"confusion_matrix": wandb.plot.confusion_matrix(
    probs=None,
    y_true=labels,
    preds=preds,
    class_names=sorted_labels,
)})

# ──────────────────────────────────────────────
# SAVE & PUSH
# ──────────────────────────────────────────────
save_dir = "./indic-SLID/best"
slid_model.save_pretrained(save_dir)
print(f"Model saved to {save_dir}")

slid_model.push_to_hub("radioduran/indic-language-identification")

Current time: 20260302_020451
torch.cuda.is_available(): True
torch.cuda.get_device_name(): NVIDIA L4
Logged in to Hugging Face!


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: dmtsss (nnti-team) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Augmentation device: cuda


preprocessor_config.json:   0%|          | 0.00/227 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/679 [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/382M [00:00<?, ?B/s]

... (18 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
print("Disconnecting...")
from google.colab import runtime
runtime.unassign()